# Ablation Study on `ap_burst2k`

Replaces `07_ablations_and_attribution.ipynb`, which runs on `data/raw/llm_generated`. That
notebook's section 3 is the source of the claim that a gap of 0.2271 is attributable to
relational and temporal modelling -- **a claim that is refuted**, because a stricter version
of the same probe (a logistic regression on one record's embedding, no pooling, no edges, no
temporal order) also scores AUROC 1.0000 on that partition. No gap remains to attribute.

An ablation is only meaningful on a benchmark where the full model is not already saturated.
On the withdrawn benchmark every variant scored 1.0000, so nothing could be ablated. That is
why these had to be re-run.

**Each ablation changes exactly one thing** and trains through the same `Trainer` as the
detector: same loss, optimiser, batch size, augmentation, epoch budget and validation-AUROC
checkpoint selection.

| Ablation | What changes | Parameters |
|---|---|---|
| baseline | -- | 670,915 |
| seed 1 | random seed only | 670,915 |
| gated readout | attention + zero-init mean/max/count gates | 671,686 |
| mean-pool readout | `AttentionPool` -> `global_mean_pool` | 670,915 |
| no structural features | 4 per-node columns removed | fewer |
| temporal removed | `NodeMemoryModule` + `TemporalAggregator` gone | 306,243 |

## 1. Setup

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import yaml
from sklearn.metrics import roc_auc_score, average_precision_score

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
import os
os.chdir(REPO_ROOT)

from src.experiments.sequence_eval import boot_ci, paired_boot

DATA_DIR = REPO_ROOT / "data" / "raw" / "ap_burst2k"
EXP = REPO_ROOT / "experiments"
BASE = EXP / "run_burst2k_base"
CONFIG = yaml.safe_load(open(REPO_ROOT / "configs" / "default.yaml"))

# Every model is scored on the SAME 500 test sequences, in the same order. Each run's
# sequence_scores_test.npz carries its own sequence_id and y, and they are asserted equal
# before anything is paired -- comparing two models scored on different orderings produces a
# number that looks fine and means nothing.
base = np.load(BASE / "sequence_scores_test.npz", allow_pickle=True)
Y = base["y"]
ORDER = base["sequence_id"]
DETECTOR = base["T-GNN_baseline_(window_max)"]
N_BOOT = 2000

print(f"{len(Y)} test sequences, {int(Y.sum())} poisoned")
print(f"detector (T-GNN, window max): AUROC {roc_auc_score(Y, DETECTOR):.4f}  "
      f"AP {average_precision_score(Y, DETECTOR):.4f}")


def load_scores(run_dir, key_filter=None, filename="sequence_scores_test.npz"):
    """Per-sequence scores from a run, with the ordering checked against the detector's.

    key_filter matters more than it looks. Every run directory written by sequence_eval also
    contains that run's copy of the LOGISTIC REGRESSION baselines, because sequence_eval
    computes them every time. Loading a static-GNN run without filtering therefore pulls in
    three regression rows and labels them as static-GNN results.
    """
    p = Path(run_dir) / filename
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=True)
    assert (z["sequence_id"] == ORDER).all(), f"{p} is ordered differently -- cannot pair"
    assert (z["y"] == Y).all(), f"{p} has different labels -- different split?"
    return {k: z[k] for k in z.files
            if k not in ("sequence_id", "y") and (key_filter is None or key_filter in k)}


def row(name, scores, expected=None, pair=True):
    auroc, apr = roc_auc_score(Y, scores), average_precision_score(Y, scores)
    lo, hi = boot_ci(Y, scores, N_BOOT)
    out = f"{name:<40} {auroc:>7.4f} {apr:>7.4f}  [{lo:.4f}, {hi:.4f}]"
    if pair:
        d, dlo, dhi = paired_boot(Y, DETECTOR, scores, N_BOOT, metric="auroc")
        sig = "*" if (dlo > 0 or dhi < 0) else " "
        out += f"  {d:+.4f}{sig}"
    if expected:
        out += f"   exp {expected[0]:.4f}/{expected[1]:.4f}"
    print(out)
    return auroc, apr


HEAD = (f"{'model':<40} {'AUROC':>7} {'AP':>7}  {'95% CI':>18}  "
        f"{'dAUROC':>8}\n" + "-" * 104)

## 2. Seed stability -- read this before any other ablation

An ablation difference is only meaningful against the run-to-run noise floor. Two independent
seeds bound it, and re-running evaluation on the *same* checkpoint in a fresh process moves
graph AUROC by about +/-0.007.

**Treat any difference below about 0.02 as noise.** Several ablations below fall inside that
band, and reporting them as effects would be wrong.

In [ ]:
print(HEAD)
for label, run, exp in (("seed 42 (baseline)", BASE, (0.8122, 0.8185)),
                        ("seed 1", EXP / "run_burst2k_base_s1", (0.8140, 0.8204))):
    s = load_scores(run, key_filter="window_max")
    if not s:
        print(f"{label:<40} PENDING")
        continue
    row(label, list(s.values())[0], exp, pair=(run != BASE))
print("\nspread across seeds: 0.0018 AUROC. That is the noise floor -- nothing below")
print("about 0.02 should be reported as an effect.")

## 3. Readout ablations

Three ways to pool node embeddings into a graph embedding.

- **attention** -- the published baseline. MIL-style attention pooling (Ilse et al., 2018).
- **gated** -- attention plus zero-initialised mean/max/count residual branches. Bit-identical
  to attention at initialisation, so it *starts* as the baseline and departs only insofar as
  training opens a gate. On `ap_burst2k` the learned gates ended at
  `[+0.0004, -0.0023, +0.0186]`: the model was free to use magnitude information and largely
  declined.
- **mean-pool** -- `AttentionPool` replaced by `global_mean_pool`, nothing else changed. This
  is the direct test of the Sections 2.6 and 3.8 claim that mean pooling averages a small
  anomalous subgraph away by construction. **It had never been measured on this benchmark.**

In [ ]:
print("produced by:")
print("  gated    : python -m src.models.train --data data/raw/ap_burst2k \\")
print("               --output experiments/run_burst2k_gated --readout gated ...")
print("  mean-pool: python -m src.experiments.train_ablation_meanpool \\")
print("               --data data/raw/ap_burst2k --output experiments/run_burst2k_meanpool \\")
print("               --structural-features --skip-phase1 --epochs-supervised 12 --seed 42\n")
print(HEAD)
for label, run, exp in (("attention (baseline)", BASE, (0.8122, 0.8185)),
                        ("gated readout", EXP / "run_burst2k_gated", (0.8016, 0.8095)),
                        ("mean-pool readout", EXP / "run_burst2k_meanpool", None)):
    s = load_scores(run, key_filter="window_max")
    if not s:
        print(f"{label:<40} PENDING -- see the command above")
        continue
    row(label, list(s.values())[0], exp, pair=(run != BASE))

## 4. Input-feature ablation: structural node features

The dataset optionally emits four extra per-node columns on top of the 384-d embedding and 3
metadata columns: degree under two scalings, and the mean and max weight of a node's incident
edges. Every reported result trains with them enabled.

Removing them changes the model's input width, so the checkpoint must be evaluated with
`--no-structural-features`. Evaluating it as though it had them loads the weights into a
differently-shaped model and produces a number that means nothing -- the failure recorded in
`experiments/eval_ap_scale.INVALID_train_eval_mismatch.json`.

In [ ]:
print("produced by: python -m src.models.train --data data/raw/ap_burst2k \\")
print("               --output experiments/run_burst2k_nostruct --readout attention \\")
print("               --skip-phase1 --select-by auroc --epochs-supervised 12 --seed 42")
print("  (note: no --structural-features flag)\n")
print("evaluate with: python -m src.experiments.sequence_eval --data data/raw/ap_burst2k \\")
print("                 --run-dir experiments/run_burst2k_nostruct --no-structural-features\n")
print(HEAD)
for label, run, exp in (("with structural features (baseline)", BASE, (0.8122, 0.8185)),
                        ("without structural features", EXP / "run_burst2k_nostruct", None)):
    s = load_scores(run, key_filter="window_max")
    if not s:
        print(f"{label:<40} PENDING -- see the commands above")
        continue
    row(label, list(s.values())[0], exp, pair=(run != BASE))

## 5. Component ablation: remove the temporal machinery

The largest effect in the study, and the cleanest.

`NodeMemoryModule` (the per-node GRU carrying state between snapshots) and
`TemporalAggregator` (the GRU over the snapshot sequence) are removed. The forward pass reads
`graph_sequence[-1]` and nothing else. The GAT encoder, the pooling, the head architectures,
the input width, the loss and the training recipe are all unchanged.

Run under both readouts so the loss cannot be attributed to pooling. The `window last`
variants land at 0.4813 and 0.4860 -- indistinguishable from chance, which is exactly what
the benchmark's construction predicts for a model that reads one snapshot.

In [ ]:
print(HEAD)
for tag, readout, exp in (("attn", "attention", (0.5602, 0.4684)),
                          ("mean", "mean", (0.5424, 0.4652))):
    s = load_scores(EXP / f"run_burst2k_static_{tag}")
    if not s:
        print(f"{'temporal removed (' + readout + ')':<40} PENDING")
        continue
    for agg in ("max", "mean", "last"):
        key = f"static_GNN_({readout})_(window_{agg})"
        if key in s:
            row(f"temporal removed, {readout}, {agg}", s[key], exp if agg == "max" else None)

## 6. Ablation summary

In [ ]:
runs = [("attention readout (baseline)", BASE, "window_max"),
        ("seed 1", EXP / "run_burst2k_base_s1", "window_max"),
        ("gated readout", EXP / "run_burst2k_gated", "window_max"),
        ("mean-pool readout", EXP / "run_burst2k_meanpool", "window_max"),
        ("no structural features", EXP / "run_burst2k_nostruct", "window_max"),
        ("temporal removed (attention)", EXP / "run_burst2k_static_attn", "window_max"),
        ("temporal removed (mean)", EXP / "run_burst2k_static_mean", "window_max")]

base_auroc = roc_auc_score(Y, DETECTOR)
print(f"{'ablation':<36} {'AUROC':>7} {'AP':>7} {'d vs baseline':>15}  verdict")
print("-" * 88)
pending = []
for label, run, filt in runs:
    s = load_scores(run, key_filter=filt)
    if not s:
        pending.append(label)
        print(f"{label:<36} {'PENDING':>7}")
        continue
    v = list(s.values())[0]
    auroc, apr = roc_auc_score(Y, v), average_precision_score(Y, v)
    d = auroc - base_auroc
    if run == BASE:
        verdict = "reference"
    elif abs(d) < 0.02:
        verdict = "within noise (|d| < 0.02)"
    else:
        dd, lo, hi = paired_boot(Y, DETECTOR, v, N_BOOT, metric="auroc")
        verdict = "SIGNIFICANT" if (lo > 0 or hi < 0) else "not significant"
    print(f"{label:<36} {auroc:>7.4f} {apr:>7.4f} {d:>+15.4f}  {verdict}")

if pending:
    print(f"\n{len(pending)} ablation(s) not yet measured: {', '.join(pending)}")

print("""
WHAT THIS SHOWS

Removing the temporal components costs about 0.25 AUROC and 0.35 AP, under either readout.
That is the study's central empirical claim and it is what the benchmark was rebuilt to make
measurable.

Readout choice and seed move the result by less than the 0.02 noise floor. Do not report
those as effects -- report them as evidence the headline is stable.
""")